In [1]:
import os
import ast

import pandas as pd
import matplotlib.pyplot as plt
import scipy

from plotting_utils import (
    get_train_domain,
    get_test_domain,
    get_model_name,
    wanted_metrics
)

pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.multi_sparse', True)
PROJECT_DIR = os.environ["MDS"]

# Load and format results

In [2]:
results_df = pd.read_csv(
    f"{PROJECT_DIR}/results_mayarr_sample/experiments_mayarr/aggregated_results.csv",
    header=[0,1]
)
results_df.columns = pd.Index([
  f"{c[0]}_{c[1]}" if c[1]!="nan" else c[0] for c in results_df.columns
])

results_df["train_domain"] = results_df.apply(get_train_domain, axis="columns")
results_df["test_domain"] = results_df.dataset.map(get_test_domain)
results_df["model_type"] = results_df.model.map(get_model_name)

# Remove validation set
results_df = results_df[results_df.split=="test"]

new_columns = [
    "model_type",
    "train_domain",
    "test_domain",
] + wanted_metrics

# Add placeholders
for c in new_columns:
    if c not in results_df.columns:
        results_df[c] = pd.NA

results_df = results_df[new_columns]

def format_entry(entry: str):
    try:
        entry = ast.literal_eval(entry)
    except ValueError as err:
        print(f"Could not convert '{entry}' as literal")
        return pd.NA
    return entry.get("mean", pd.NA)

# Display only the mean
for metric in wanted_metrics:
    results_df[metric] = results_df[metric].map(format_entry)

# Change compression so that higher score = more compressed summary
# 1 - (# sentences in summary / # sentences in source docs)
for m in ["compression_sentence", "compression_token"]:
    results_df[m] = 1 - results_df[m]


# Hack
results_df = results_df.fillna(-1)

Could not convert '{'mean': 1.0, 'std': 0.0, 'ci': (nan, nan), 'ci_std': 0.0}' as literal
Could not convert '{'mean': 1.0, 'std': 0.0, 'ci': (nan, nan), 'ci_std': 0.0}' as literal
Could not convert '{'mean': 1.0, 'std': 0.0, 'ci': (nan, nan), 'ci_std': 0.0}' as literal
Could not convert '{'mean': 1.0, 'std': 0.0, 'ci': (nan, nan), 'ci_std': 0.0}' as literal
Could not convert '{'mean': 1.0, 'std': 0.0, 'ci': (nan, nan), 'ci_std': 0.0}' as literal
Could not convert '{'mean': 1.0, 'std': 0.0, 'ci': (nan, nan), 'ci_std': 0.0}' as literal
Could not convert '{'mean': 1.0, 'std': 0.0, 'ci': (nan, nan), 'ci_std': 0.0}' as literal
Could not convert '{'mean': 1.0, 'std': 0.0, 'ci': (nan, nan), 'ci_std': 0.0}' as literal


/tmp/ipykernel_3289354/2072983755.py:48: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  results_df = results_df.fillna(-1)


In [3]:
results_df

,model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
0,ground_truth,Convo,Convo,0.71,3.07,3.06,0.99,3.23,-1.00,-1.00,-1.00,0.89,3.30,0.73,3.39,0.90,0.85,2.57
1,PRIMERA,Science,Convo,0.80,1.43,1.49,0.41,1.77,0.19,0.11,-0.02,0.64,1.34,0.67,1.28,-4.46,-2.61,1.58
2,PRIMERA,Convo,Convo,0.81,1.41,1.52,0.40,1.77,0.19,0.11,-0.01,0.64,1.31,0.75,1.22,-5.10,-3.93,1.54
3,PRIMERA,News+,Convo,0.82,1.34,1.53,0.40,1.76,0.19,0.11,-0.01,0.67,1.36,0.72,1.26,-5.29,-3.17,1.52
4,SummN,Science,Convo,0.39,1.13,1.21,0.33,1.26,0.21,0.14,-0.04,0.63,1.37,0.33,1.25,-6.52,-4.14,1.26
5,SummN,Convo,Convo,0.66,2.39,2.50,0.72,2.71,0.33,0.21,0.09,0.91,2.88,0.71,2.78,-10.81,-8.11,2.18
6,SummN,News+,Convo,0.67,1.44,1.58,0.71,1.95,0.22,0.13,0.00,0.70,1.74,0.73,1.58,-3.45,-2.16,1.67
7,ConvoSumm,Science,Convo,0.44,1.22,1.26,0.26,1.34,0.22,0.15,-0.01,0.60,1.27,0.26,1.29,-9.32,-7.78,1.25
8,ConvoSumm,Convo,Convo,0.64,2.13,2.39,0.70,2.52,0.31,0.21,0.06,0.91,2.61,0.59,2.43,-12.40,-12.23,1.97
9,ConvoSumm,News+,Convo,0.68,1.59,1.78,0.75,1.97,0.23,0.13,0.01,0.70,1.72,0.78,1.74,-3.62,-2.51,1.77


In [4]:
a = results_df.set_index(["model_type", "train_domain", "test_domain"]).sort_index()
print(a.to_csv(float_format='%.2f'))
a

model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,Convo,Convo,0.64,2.13,2.39,0.70,2.52,0.31,0.21,0.06,0.91,2.61,0.59,2.43,-12.40,-12.23,1.97
ConvoSumm,Convo,News+,0.75,1.83,1.90,0.51,2.08,0.24,0.14,0.00,0.91,2.02,0.57,1.94,-4.31,-7.54,1.68
ConvoSumm,Convo,Science,0.67,1.95,1.97,0.61,2.27,0.23,0.14,0.02,0.91,2.22,0.58,2.03,-5.17,-9.23,1.76
ConvoSumm,News+,Convo,0.68,1.59,1.78,0.75,1.97,0.23,0.13,0.01,0.70,1.72,0.78,1.74,-3.62,-2.51,1.77
ConvoSumm,News+,News+,0.78,2.39,2.54,0.80,2.72,0.46,0.24,0.20,0.74,2.68,0.84,2.99,-1.31,-1.45,2.28
ConvoSumm,News+,Science,0.69,1.77,1.76,0.70,2.13,0.23,0.13,0.04,0.81,1.79,0.71,1.87,-2.31,-2.32,1.89
ConvoSumm,Science,Convo,0.44,1.22,1.26,0.26,1.34,0.22,0.15,-0.01,0.60,1.27,0.26,1.29,-9.

unieval_factuality  \
model_type         train_domain test_domain                       
ConvoSumm          Convo        Convo                      0.64   
                                News+                      0.75   
                                Science                    0.67   
                   News+        Convo                      0.68   
                                News+                      0.78   
                                Science                    0.69   
                   Science      Convo                      0.44   
                                News+                      0.66   
                                Science                    0.65   
Llama 3.1          Convo        Convo                      0.81   
                                News+                      0.87   
                                Science                    0.88   
                   News+        Convo                      0.82   
                                News+                      0.87   
                                Science                    0.88   
                   Science      Convo                      0.82   
                                News+                      0.87   
                                Science                    0.88   
Llama 3.1 (0-shot) zero-shot    Convo                      0.83   
                                News+                      0.88   
                                Science                    0.87   
PRIMERA            Convo        Convo                      0.81   
                                News+                      0.83   
                                Science                    0.84   
                   News+        Convo                      0.82   
                                News+                      0.78   
                                Science                    0.82   
                   Science      Convo                      0.80   
                                News+                      0.78   
                                Science                    0.84   
SummN              Convo        Convo                      0.66   
                                News+                      0.90   
                                Science                    0.77   
                   News+        Convo                      0.67   
                                News+                      0.72   
                                Science                    0.64   
                   Science      Convo                      0.39   
                                News+                      0.65   
                                Science                    0.56   
ground_truth       Convo        Convo                      0.71   
                   News+        News+                      0.73   
                   Science      Science                    0.55   

                                             prometheus_factuality  \
model_type         train_domain test_domain                          
ConvoSumm          Convo        Convo                         2.13   
                                News+                         1.83   
                                Science                       1.95   
                   News+        Convo                         1.59   
                                News+                         2.39   
                                Science                       1.77   
                   Science      Convo                         1.22   
                                News+                         1.81   
                                Science                       1.61   
Llama 3.1          Convo        Convo                         3.09   
                                News+                         3.24   
                                Science                       3.32   
                   News+        Convo                         3.01   
                                News+         

In [5]:
a = results_df[results_df.model_type=="PRIMERA"].set_index(["model_type", "train_domain", "test_domain"]).sort_index()
a.transpose()

model_type             PRIMERA                                            \
train_domain             Convo               News+               Science   
test_domain              Convo News+ Science Convo News+ Science   Convo   
unieval_factuality        0.81  0.83    0.84  0.82  0.78    0.82    0.80   
prometheus_factuality     1.41  1.97    1.96  1.34  2.01    2.03    1.43   
prometheus_consistency    1.52  2.08    1.95  1.53  2.17    1.96    1.49   
unieval_relevance         0.40  0.41    0.61  0.40  0.43    0.65    0.41   
prometheus_relevance      1.77  2.38    2.38  1.76  2.37    2.40    1.77   
rouge_rouge1              0.19  0.38    0.23  0.19  0.37    0.23    0.19   
rouge_rougeLsum           0.11  0.18    0.12  0.11  0.18    0.12    0.11   
moverscore               -0.01  0.13    0.06 -0.01  0.12    0.06   -0.02   
unieval_fluency           0.64  0.63    0.79  0.67  0.63    0.82    0.64   
prometheus_fluency        1.31  2.04    1.82  1.36  1.94    1.92    1.34   
unieval_coherence         0.75  0.71    0.73  0.72  0.65    0.74    0.67   
prometheus_coherence      1.22  1.94    1.79  1.26  1.86    1.83    1.28   
compression_sentence     -5.10 -1.99   -2.23 -5.29 -2.33   -2.40   -4.46   
compression_token        -3.93 -1.28   -1.59 -3.17 -1.38   -1.61   -2.61   
prometheus_coverage       1.54  2.01    1.94  1.52  1.94    2.04    1.58   

model_type                            
train_domain                          
test_domain            News+ Science  
unieval_factuality      0.78    0.84  
prometheus_factuality   1.96    1.96  
prometheus_consistency  2.07    1.98  
unieval_relevance       0.44    0.62  
prometheus_relevance    2.33    2.39  
rouge_rouge1            0.38    0.22  
rouge_rougeLsum         0.18    0.12  
moverscore              0.13    0.05  
unieval_fluency         0.62    0.77  
prometheus_fluency      1.95    1.75  
unieval_coherence       0.68    0.74  
prometheus_coherence    1.94    1.80  
compression_sentence   -2.42   -2.42  
compression_token      -1.51   -1.67  
prometheus_coverage     1.99    2.00

# Q: How well is each model trained _on its domain_?


In [6]:
# Limit to models where the model is trained and tested on the same domain
same_domain_analysis = results_df[
    results_df.apply(lambda x: x.train_domain==x.test_domain, axis="columns")
].sort_values(by=["train_domain", "test_domain"])
same_domain_analysis = same_domain_analysis.loc[:, ["train_domain", "test_domain", "model_type"]+wanted_metrics]

same_domain_performance_dict = same_domain_analysis.set_index(["train_domain", "model_type"])[wanted_metrics].to_dict(orient="index")
same_domain_performance_dict

{('Convo', 'ground_truth'): {'unieval_factuality': 0.7088183022881179,
  'prometheus_factuality': 3.07,
  'prometheus_consistency': 3.06,
  'unieval_relevance': 0.9873195765236236,
  'prometheus_relevance': 3.23,
  'rouge_rouge1': -1.0,
  'rouge_rougeLsum': -1.0,
  'moverscore': -1.0,
  'unieval_fluency': 0.8947679057879945,
  'prometheus_fluency': 3.3,
  'unieval_coherence': 0.7266774421924654,
  'prometheus_coherence': 3.39,
  'compression_sentence': 0.899399009312196,
  'compression_token': 0.8506645011781512,
  'prometheus_coverage': 2.57},
 ('Convo', 'PRIMERA'): {'unieval_factuality': 0.8115564810063642,
  'prometheus_factuality': 1.41,
  'prometheus_consistency': 1.52,
  'unieval_relevance': 0.40201506659159114,
  'prometheus_relevance': 1.77,
  'rouge_rouge1': 0.18966978955204078,
  'rouge_rougeLsum': 0.10710250375191789,
  'moverscore': -0.014320818879059666,
  'unieval_fluency': 0.6432680571368042,
  'prometheus_fluency': 1.31,
  'unieval_coherence': 0.7488849508238488,
  'pro

In [7]:
a = same_domain_analysis.set_index(["model_type"]).sort_index()
print(a.to_csv())
a

model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,Convo,Convo,0.640659595215039,2.13,2.39,0.7038211621316407,2.52,0.3107453771812775,0.212665444995344,0.0602550780555246,0.9053736660495955,2.61,0.5918988386538738,2.43,-12.401666666666667,-12.227228951576869,1.97
ConvoSumm,News+,News+,0.7783237624298042,2.39,2.54,0.8013146991897413,2.72,0.45739665892896947,0.2353538463751893,0.19886831916532757,0.7410538537314413,2.68,0.8442774236204191,2.99,-1.3133591988077282,-1.4453698077933264,2.28
ConvoSumm,Science,Science,0.6474810323495521,1.61,1.49,0.4902085290006365,1.84,0.269789839482931,0.15840737749212283,0.034695989039083434,0.6787635389357702,1.6,0.3504087520182354,1.51,-4.905978021978022,-6.821297877632365,1.53
Llama 3.1,

,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
model_type,,,,,,,,,,,,,,,,,
ConvoSumm,Convo,Convo,0.64,2.13,2.39,0.70,2.52,0.31,0.21,0.06,0.91,2.61,0.59,2.43,-12.40,-12.23,1.97
ConvoSumm,News+,News+,0.78,2.39,2.54,0.80,2.72,0.46,0.24,0.20,0.74,2.68,0.84,2.99,-1.31,-1.45,2.28
ConvoSumm,Science,Science,0.65,1.61,1.49,0.49,1.84,0.27,0.16,0.03,0.68,1.60,0.35,1.51,-4.91,-6.82,1.53
Llama 3.1,Convo,Convo,0.81,3.09,3.26,0.95,3.47,0.21,0.15,0.05,0.93,3.94,0.96,3.97,-3.64,-1.35,2.95
Llama 3.1,News+,News+,0.87,3.20,3.47,0.92,3.59,0.40,0.21,0.16,0.94,4.04,0.95,3.93,-2.23,-1.75,2.96
Llama 3.1,Science,Science,0.88,3.40,3.46,0.91,3.53,0.24,0.15,0.08,0.93,3.94,0.97,3.95,-1.38,-1.29,3.17
PRIMERA,Convo,Convo,0.81,1.41,1.52,0.40,1.77,0.19,0.11,-0.01,0.64,1.31,0.75,1.22,-5.10,-3.93,1.54
PRIMERA,News+,News+,0.78,2.01,2.17,0.43,2.37,0.37,0.18,0.12,0.63,1.94,0.65,1.86,-2.33,-1.38,1.94
PRIMERA,Science,Science,0.84,1.96,1.98,0.62,2.39,0.22,0.12,0.05,0.77,1.75,0.74,1.80,-2.42,-1.67,2.00


In [8]:
a = same_domain_analysis.groupby(["model_type"]).mean(numeric_only=True).to_csv(float_format='%.2f')
print(a)

model_type,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,0.69,2.04,2.14,0.67,2.36,0.35,0.20,0.10,0.78,2.30,0.60,2.31,-6.21,-6.83,1.93
Llama 3.1,0.85,3.23,3.40,0.93,3.53,0.28,0.17,0.10,0.93,3.97,0.96,3.95,-2.41,-1.46,3.03
PRIMERA,0.81,1.79,1.89,0.49,2.18,0.26,0.14,0.05,0.68,1.67,0.71,1.63,-3.28,-2.32,1.83
SummN,0.65,2.02,2.08,0.66,2.33,0.35,0.20,0.10,0.78,2.26,0.63,2.31,-5.33,-4.68,1.95
ground_truth,0.66,2.58,2.69,0.99,3.13,-1.00,-1.00,-0.33,0.78,2.53,0.64,2.87,-2.20,-1.96,2.39



In [9]:
a = same_domain_analysis[same_domain_analysis.model_type!='ground_truth'].groupby(["train_domain"]).mean(numeric_only=True)
print(a.to_csv(float_format='%.2f'))
a

train_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
Convo,0.73,2.25,2.42,0.69,2.62,0.26,0.17,0.05,0.85,2.69,0.75,2.60,-7.99,-6.40,2.16
News+,0.79,2.46,2.62,0.73,2.81,0.42,0.21,0.17,0.76,2.79,0.81,2.89,-1.76,-1.45,2.33
Science,0.73,2.11,2.09,0.63,2.37,0.25,0.15,0.05,0.77,2.17,0.61,2.17,-3.17,-3.62,2.05



,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
train_domain,,,,,,,,,,,,,,,
Convo,0.73,2.25,2.42,0.69,2.62,0.26,0.17,0.05,0.85,2.69,0.75,2.60,-7.99,-6.40,2.16
News+,0.79,2.46,2.62,0.73,2.81,0.42,0.21,0.17,0.76,2.79,0.81,2.89,-1.76,-1.45,2.33
Science,0.73,2.11,2.09,0.63,2.37,0.25,0.15,0.05,0.77,2.17,0.61,2.17,-3.17,-3.62,2.05


# Q: Which train domain transfers the best to other domains?

Limit to models that are trained and tested on _different_ domains. Also, focus on performance _loss_ versus absolute performance. 


In [10]:
cross_performance = results_df[results_df.train_domain!=results_df.test_domain].groupby(["model_type", "train_domain", "test_domain"]).mean(numeric_only=True)
print(cross_performance.to_csv())
cross_performance

model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,Convo,News+,0.7465658013353306,1.83,1.9,0.5056588471175113,2.08,0.23731259476435618,0.13813155329140192,0.00041119504713410526,0.9103890092792085,2.02,0.572873641507687,1.94,-4.307876984126983,-7.537310613458084,1.68
ConvoSumm,Convo,Science,0.673443167817001,1.95,1.97,0.6132739132713839,2.27,0.22975463311411906,0.1391126410290911,0.020910987602955688,0.9134477122673242,2.22,0.583611342426411,2.03,-5.167416666666667,-9.232939385777877,1.76
ConvoSumm,News+,Convo,0.680962832314046,1.59,1.78,0.7516980356333706,1.97,0.2279826399002288,0.13159840990020666,0.008295298076173978,0.6983868840575984,1.72,0.7796710883409483,1.74,-3.6184218583703878,-2.514540225447379,1.77
ConvoSumm

unieval_factuality  \
model_type         train_domain test_domain                       
ConvoSumm          Convo        News+                      0.75   
                                Science                    0.67   
                   News+        Convo                      0.68   
                                Science                    0.69   
                   Science      Convo                      0.44   
                                News+                      0.66   
Llama 3.1          Convo        News+                      0.87   
                                Science                    0.88   
                   News+        Convo                      0.82   
                                Science                    0.88   
                   Science      Convo                      0.82   
                                News+                      0.87   
Llama 3.1 (0-shot) zero-shot    Convo                      0.83   
                                News+                      0.88   
                                Science                    0.87   
PRIMERA            Convo        News+                      0.83   
                                Science                    0.84   
                   News+        Convo                      0.82   
                                Science                    0.82   
                   Science      Convo                      0.80   
                                News+                      0.78   
SummN              Convo        News+                      0.90   
                                Science                    0.77   
                   News+        Convo                      0.67   
                                Science                    0.64   
                   Science      Convo                      0.39   
                                News+                      0.65   

                                             prometheus_factuality  \
model_type         train_domain test_domain                          
ConvoSumm          Convo        News+                         1.83   
                                Science                       1.95   
                   News+        Convo                         1.59   
                                Science                       1.77   
                   Science      Convo                         1.22   
                                News+                         1.81   
Llama 3.1          Convo        News+                         3.24   
                                Science                       3.32   
                   News+        Convo                         3.01   
                                Science                       3.38   
                   Science      Convo                         3.17   
                                News+                         3.25   
Llama 3.1 (0-shot) zero-shot    Convo                         3.12   
                                News+                         3.26   
                                Science                       3.42   
PRIMERA            Convo        News+                         1.97   
                                Science                       1.96   
                   News+        Convo                         1.34   
                                Science                       2.03   
                   Science      Convo                         1.43   
                                News+                         1.96   
SummN              Convo        News+                         2.71   
                                Science                       2.51   
                   News+        Convo                         1.44   
                                Science                       1.57   
                   Science      Convo                         1.13   
                                News+                         1.95   

                                             prometheus_consistency  \

In [11]:
cross_performance.groupby(["train_domain", "test_domain"]).mean()


unieval_factuality  prometheus_factuality  \
train_domain test_domain                                              
Convo        News+                      0.84                   2.44   
             Science                    0.79                   2.44   
News+        Convo                      0.75                   1.84   
             Science                    0.76                   2.19   
Science      Convo                      0.61                   1.74   
             News+                      0.74                   2.24   
zero-shot    Convo                      0.83                   3.12   
             News+                      0.88                   3.26   
             Science                    0.87                   3.42   

                          prometheus_consistency  unieval_relevance  \
train_domain test_domain                                              
Convo        News+                          2.54               0.62   
             Science                        2.44               0.71   
News+        Convo                          2.03               0.70   
             Science                        2.18               0.73   
Science      Convo                          1.79               0.48   
             News+                          2.38               0.49   
zero-shot    Convo                          3.31               0.94   
             News+                          3.41               0.93   
             Science                        3.52               0.93   

                          prometheus_relevance  rouge_rouge1  rouge_rougeLsum  \
train_domain test_domain                                                        
Convo        News+                        2.71          0.31             0.17   
             Science                      2.68          0.23             0.14   
News+        Convo                        2.28          0.21             0.13   
             Science                      2.57          0.23             0.13   
Science      Convo                        1.96          0.21             0.14   
             News+                        2.61          0.35             0.18   
zero-shot    Convo                        3.44          0.22             0.15   
             News+                        3.57          0.41             0.22   
             Science                      3.47          0.24             0.16   

                          moverscore  unieval_fluency  prometheus_fluency  \
train_domain test_domain                                                    
Convo        News+              0.07             0.84                2.77   
             Science            0.04             0.89                2.68   
News+        Convo              0.01             0.75                2.20   
             Science            0.05             0.83                2.34   
Science      Convo             -0.00             0.70                1.98   
             News+              0.09             0.72                2.43   
zero-shot    Convo              0.05             0.93                4.01   
             News+              0.16             0.94                4.04   
             Science            0.08             0.93                4.06   

                          unieval_coherence  prometheus_coherence  \
train_domain test_domain                                            
Convo        News+                     0.76                  2.65   
             Science                   0.79                  2.52   
News+        Convo                     0.80                  2.13   
             Science                   0.78                  2.35   
Science      Convo                     0.56                  1.94   
             News+                     0.57                  2.42   
zero-shot    Convo                     0.96                  3.87   
             News+                     0.94                  4.00   
             Science            

In [12]:
# Limit to models where the model is trained and tested on different domains
def safe_divide(num, denom):
    if denom == 0:
        return 0.0
    return num / denom

def get_relative_performance_loss(row):
    key = [row.train_domain, row.model_type]
    if row.train_domain == "zero-shot":
        key[0] = row.test_domain
        key[1] = "ICL"
    original_performance = same_domain_performance_dict.get(
        tuple(key),
        {}
    )
    rel_performance = {
        "train_domain": row.train_domain,
        "test_domain": row.test_domain,
        "model_type": row.model_type
    }
    rel_performance.update({
        m: safe_divide(
            row[m] - original_performance.get(m, 0.0), original_performance.get(m, 0.0)
        ) for m in wanted_metrics
    })
    return rel_performance

performance_diff = results_df[results_df.train_domain!=results_df.test_domain].apply(
    get_relative_performance_loss,
    axis="columns",
    result_type="expand"
)


In [13]:
diff = performance_diff.groupby(["model_type", "train_domain", "test_domain"]).mean(numeric_only=True)

print(diff.to_csv())

diff


model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,Convo,News+,0.16530807766134192,-0.14084507042253513,-0.20502092050209214,-0.28155208407482596,-0.1746031746031746,-0.23631174527202478,-0.35047485831830316,-0.9931757611075502,0.0055395285037352976,-0.22605363984674326,-0.03214264989851111,-0.2016460905349795,-0.6526372543372948,-0.38356346778915557,-0.1472081218274112
ConvoSumm,Convo,Science,0.051171593849239194,-0.0845070422535211,-0.1757322175732218,-0.1286509325551099,-0.0992063492063492,-0.26063378577603546,-0.345861566592839,-0.6529589160321663,0.008917915906432487,-0.1494252873563217,-0.014001541625441727,-0.16460905349794253,-0.5833288536487031,-0.24488701222960554,-0.10659898477157359
ConvoSumm,News+,Convo,-0.

unieval_factuality  \
model_type         train_domain test_domain                       
ConvoSumm          Convo        News+                      0.17   
                                Science                    0.05   
                   News+        Convo                     -0.13   
                                Science                   -0.11   
                   Science      Convo                     -0.32   
                                News+                      0.02   
Llama 3.1          Convo        News+                      0.07   
                                Science                    0.08   
                   News+        Convo                     -0.06   
                                Science                    0.01   
                   Science      Convo                     -0.07   
                                News+                     -0.01   
Llama 3.1 (0-shot) zero-shot    Convo                      0.00   
                                News+                      0.00   
                                Science                    0.00   
PRIMERA            Convo        News+                      0.02   
                                Science                    0.04   
                   News+        Convo                      0.05   
                                Science                    0.05   
                   Science      Convo                     -0.05   
                                News+                     -0.07   
SummN              Convo        News+                      0.36   
                                Science                    0.16   
                   News+        Convo                     -0.08   
                                Science                   -0.12   
                   Science      Convo                     -0.29   
                                News+                      0.17   

                                             prometheus_factuality  \
model_type         train_domain test_domain                          
ConvoSumm          Convo        News+                        -0.14   
                                Science                      -0.08   
                   News+        Convo                        -0.33   
                                Science                      -0.26   
                   Science      Convo                        -0.24   
                                News+                         0.12   
Llama 3.1          Convo        News+                         0.05   
                                Science                       0.07   
                   News+        Convo                        -0.06   
                                Science                       0.06   
                   Science      Convo                        -0.07   
                                News+                        -0.04   
Llama 3.1 (0-shot) zero-shot    Convo                         0.00   
                                News+                         0.00   
                                Science                       0.00   
PRIMERA            Convo        News+                         0.40   
                                Science                       0.39   
                   News+        Convo                        -0.33   
                                Science                       0.01   
                   Science      Convo                        -0.27   
                                News+                         0.00   
SummN              Convo        News+                         0.13   
                                Science                       0.05   
                   News+        Convo                        -0.35   
                                Science                      -0.29   
                   Science      Convo                        -0.23   
                                News+                         0.34   

                                             prometheus_consistency  \

In [14]:
model_display_order = [
    "ConvoSumm",
    "PRIMERA",
    "SummN",
    "Llama 3.1",
    "Llama 3.1 (0-shot)"
]
diff_data = performance_diff.groupby(["model_type"]).mean(numeric_only=True).reindex(model_display_order)
print(diff_data.to_csv())
diff_data

model_type,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
ConvoSumm,-0.053827243253098365,-0.15625113240842178,-0.14773020162235695,-0.2589360856699074,-0.14948189420695818,-0.25863342825498353,-0.26937408556980097,-0.7639393129499846,-0.020182279768257986,-0.20679623911095865,-0.11415961884082942,-0.19646080569500393,0.3104795339982407,0.07308295746058702,-0.12116255354221124
PRIMERA,0.008857699937731759,0.032240465785128535,0.009598419721509691,0.05653277109257129,0.02667016164577604,0.1384105177973956,0.09326014968646268,-2.7520538717774454,0.03330349467932115,0.08621442249678656,-0.0005420731646424704,0.08482803830062774,0.16152520360486927,0.11170747555073374,0.030831101887802897
SummN,0.03480789450697589,-0.0584091141063687,-0.06503327283726554,-0.1692823

,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
model_type,,,,,,,,,,,,,,,
ConvoSumm,-0.05,-0.16,-0.15,-0.26,-0.15,-0.26,-0.27,-0.76,-0.02,-0.21,-0.11,-0.20,0.31,0.07,-0.12
PRIMERA,0.01,0.03,0.01,0.06,0.03,0.14,0.09,-2.75,0.03,0.09,-0.00,0.08,0.16,0.11,0.03
SummN,0.03,-0.06,-0.07,-0.17,-0.06,-0.30,-0.28,-0.90,-0.02,-0.05,-0.02,-0.11,0.29,0.14,-0.06
Llama 3.1,0.01,0.00,0.01,0.00,0.01,0.13,0.05,0.34,0.00,0.02,0.00,0.00,0.30,0.04,0.01
Llama 3.1 (0-shot),0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [15]:
performance_diff.groupby(["train_domain"]).mean(numeric_only=True)

,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
train_domain,,,,,,,,,,,,,,,
Convo,0.12,0.11,0.05,-0.01,0.05,0.13,-0.01,-1.98,0.03,0.07,0.03,0.06,-0.52,-0.15,0.04
News+,-0.05,-0.20,-0.21,0.00,-0.15,-0.47,-0.38,-0.80,0.05,-0.21,-0.02,-0.24,0.78,0.39,-0.11
Science,-0.08,-0.05,0.01,-0.27,-0.04,0.12,0.09,-0.28,-0.08,0.02,-0.11,0.02,0.54,0.03,-0.04
zero-shot,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [16]:
diff_data = performance_diff.groupby(["train_domain", "test_domain"]).mean(numeric_only=True)
print(diff_data.to_csv(float_format='%.2f'))
diff_data

train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
Convo,News+,0.16,0.11,0.08,-0.10,0.06,0.34,0.12,-2.50,-0.01,0.10,0.01,0.09,-0.53,-0.18,0.05
Convo,Science,0.08,0.11,0.03,0.09,0.05,-0.07,-0.13,-1.46,0.07,0.04,0.05,0.03,-0.51,-0.12,0.04
News+,Convo,-0.05,-0.27,-0.24,-0.05,-0.21,-0.49,-0.39,-0.93,-0.01,-0.24,-0.01,-0.29,1.42,0.64,-0.17
News+,Science,-0.04,-0.12,-0.18,0.06,-0.09,-0.45,-0.37,-0.66,0.11,-0.17,-0.04,-0.19,0.14,0.13,-0.05
Science,Convo,-0.18,-0.20,-0.16,-0.28,-0.20,-0.17,-0.08,-1.29,-0.10,-0.12,-0.13,-0.14,1.07,0.18,-0.16
Science,News+,0.03,0.10,0.18,-0.27,0.13,0.41,0.25,0.74,-0.06,0.16,-0.09,0.17,0.01,-0.11,0.07
zero-shot,Convo,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
zero-shot,News+,0.00,0.00,0

unieval_factuality  prometheus_factuality  \
train_domain test_domain                                              
Convo        News+                      0.16                   0.11   
             Science                    0.08                   0.11   
News+        Convo                     -0.05                  -0.27   
             Science                   -0.04                  -0.12   
Science      Convo                     -0.18                  -0.20   
             News+                      0.03                   0.10   
zero-shot    Convo                      0.00                   0.00   
             News+                      0.00                   0.00   
             Science                    0.00                   0.00   

                          prometheus_consistency  unieval_relevance  \
train_domain test_domain                                              
Convo        News+                          0.08              -0.10   
             Science                        0.03               0.09   
News+        Convo                         -0.24              -0.05   
             Science                       -0.18               0.06   
Science      Convo                         -0.16              -0.28   
             News+                          0.18              -0.27   
zero-shot    Convo                          0.00               0.00   
             News+                          0.00               0.00   
             Science                        0.00               0.00   

                          prometheus_relevance  rouge_rouge1  rouge_rougeLsum  \
train_domain test_domain                                                        
Convo        News+                        0.06          0.34             0.12   
             Science                      0.05         -0.07            -0.13   
News+        Convo                       -0.21         -0.49            -0.39   
             Science                     -0.09         -0.45            -0.37   
Science      Convo                       -0.20         -0.17            -0.08   
             News+                        0.13          0.41             0.25   
zero-shot    Convo                        0.00          0.00             0.00   
             News+                        0.00          0.00             0.00   
             Science                      0.00          0.00             0.00   

                          moverscore  unieval_fluency  prometheus_fluency  \
train_domain test_domain                                                    
Convo        News+             -2.50            -0.01                0.10   
             Science           -1.46             0.07                0.04   
News+        Convo             -0.93            -0.01               -0.24   
             Science           -0.66             0.11               -0.17   
Science      Convo             -1.29            -0.10               -0.12   
             News+              0.74            -0.06                0.16   
zero-shot    Convo              0.00             0.00                0.00   
             News+              0.00             0.00                0.00   
             Science            0.00             0.00                0.00   

                          unieval_coherence  prometheus_coherence  \
train_domain test_domain                                            
Convo        News+                     0.01                  0.09   
             Science                   0.05                  0.03   
News+        Convo                    -0.01                 -0.29   
             Science                  -0.04                 -0.19   
Science      Convo                    -0.13                 -0.14   
             News+                    -0.09                  0.17   
zero-shot    Convo                     0.00                  0.00   
             News+                     0.00                  0.00   
             Science            

In [17]:
a = performance_diff.groupby(["train_domain"]).mean(numeric_only=True)
print(a.to_csv(float_format='%.2f'))
a

train_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
Convo,0.12,0.11,0.05,-0.01,0.05,0.13,-0.01,-1.98,0.03,0.07,0.03,0.06,-0.52,-0.15,0.04
News+,-0.05,-0.20,-0.21,0.00,-0.15,-0.47,-0.38,-0.80,0.05,-0.21,-0.02,-0.24,0.78,0.39,-0.11
Science,-0.08,-0.05,0.01,-0.27,-0.04,0.12,0.09,-0.28,-0.08,0.02,-0.11,0.02,0.54,0.03,-0.04
zero-shot,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00



,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
train_domain,,,,,,,,,,,,,,,
Convo,0.12,0.11,0.05,-0.01,0.05,0.13,-0.01,-1.98,0.03,0.07,0.03,0.06,-0.52,-0.15,0.04
News+,-0.05,-0.20,-0.21,0.00,-0.15,-0.47,-0.38,-0.80,0.05,-0.21,-0.02,-0.24,0.78,0.39,-0.11
Science,-0.08,-0.05,0.01,-0.27,-0.04,0.12,0.09,-0.28,-0.08,0.02,-0.11,0.02,0.54,0.03,-0.04
zero-shot,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00


In [18]:
a = performance_diff.groupby(["test_domain"]).mean(numeric_only=True)
print(a.to_csv(float_format='%.2f'))
a

test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
Convo,-0.10,-0.21,-0.18,-0.15,-0.18,-0.29,-0.21,-0.99,-0.05,-0.16,-0.06,-0.19,1.11,0.36,-0.14
News+,0.08,0.09,0.12,-0.16,0.08,0.33,0.16,-0.78,-0.03,0.11,-0.04,0.12,-0.23,-0.13,0.05
Science,0.02,-0.01,-0.07,0.07,-0.02,-0.23,-0.23,-0.94,0.08,-0.06,0.01,-0.07,-0.17,0.00,-0.00



,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
test_domain,,,,,,,,,,,,,,,
Convo,-0.10,-0.21,-0.18,-0.15,-0.18,-0.29,-0.21,-0.99,-0.05,-0.16,-0.06,-0.19,1.11,0.36,-0.14
News+,0.08,0.09,0.12,-0.16,0.08,0.33,0.16,-0.78,-0.03,0.11,-0.04,0.12,-0.23,-0.13,0.05
Science,0.02,-0.01,-0.07,0.07,-0.02,-0.23,-0.23,-0.94,0.08,-0.06,0.01,-0.07,-0.17,0.00,-0.00


# Q: Evaluating the metrics

Theoretically, the metrics should all be very high on the ground truth datasets. Is that the case?

In [19]:
a = results_df[results_df.model_type=="ground_truth"]

print(a.to_csv(float_format='%.2f'))
a

,model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
0,ground_truth,Convo,Convo,0.71,3.07,3.06,0.99,3.23,-1.00,-1.00,-1.00,0.89,3.30,0.73,3.39,0.90,0.85,2.57
14,ground_truth,Science,Science,0.55,1.82,1.86,0.99,2.78,-1.00,-1.00,-1.00,0.67,1.53,0.45,1.51,-6.32,-5.63,1.90
28,ground_truth,News+,News+,0.73,2.84,3.16,0.99,3.39,-1.00,-1.00,1.00,0.77,2.75,0.74,3.70,-1.17,-1.10,2.69



,model_type,train_domain,test_domain,unieval_factuality,prometheus_factuality,prometheus_consistency,unieval_relevance,prometheus_relevance,rouge_rouge1,rouge_rougeLsum,moverscore,unieval_fluency,prometheus_fluency,unieval_coherence,prometheus_coherence,compression_sentence,compression_token,prometheus_coverage
0,ground_truth,Convo,Convo,0.71,3.07,3.06,0.99,3.23,-1.00,-1.00,-1.00,0.89,3.30,0.73,3.39,0.90,0.85,2.57
14,ground_truth,Science,Science,0.55,1.82,1.86,0.99,2.78,-1.00,-1.00,-1.00,0.67,1.53,0.45,1.51,-6.32,-5.63,1.90
28,ground_truth,News+,News+,0.73,2.84,3.16,0.99,3.39,-1.00,-1.00,1.00,0.77,2.75,0.74,3.70,-1.17,-1.10,2.69
